# LinOp tutorial

This notebook introduces the `LinOp` abstraction in `spacecore`.

It explains:

- what a linear operator represents,
- what role `LinOp` plays in the library,
- how a linear operator is tied to domain and codomain spaces,
- how to define and use each implemented operator type.

Current implemented operator types:

- `DenseLinOp`
- `SparseLinOp`
- `BlockDiagonalLinOp`
- `StackedLinOp`
- `SumToSingleLinOp`


## 1. What a `LinOp` signifies

A `LinOp` is the library abstraction that represents a linear map between two spaces.

Let $A$ be a linear map

$$
A : X \to Y,
$$

then `LinOp` is the object that encodes this map together with the numerical policy under which it is applied.

So a linear operator is not just an array or a matrix. It is a map equipped with:

- a **domain** space,
- a **codomain** space,
- a **context**,
- methods for forward and adjoint application.

This means that `LinOp` makes the geometry explicit:

$$
x \in X \mapsto Ax \in Y.
$$

Thus, the abstraction is richer than raw matrix storage. It knows what kind of objects it acts on and what kind of objects it returns.


Schematically, the role split is

$$
\texttt{BackendOps} \to \texttt{Context} \to \texttt{Space} \to \texttt{LinOp}.
$$

- `BackendOps` says how numerical computations are performed.
- `Context` says which backend, dtype, and checking policy are active.
- `Space` says what kind of mathematical objects we are working with.
- `LinOp` says how one space is mapped linearly into another.


## 2. Core ingredients of a linear operator

A linear operator in the library is organized around three objects:

$$
(X, Y, A),
$$

where

- `X` is the domain,
- `Y` is the codomain,
- `A` is the linear map.

The library stores the first two explicitly as spaces:

- `op.dom` - domain (i.e., $X$)
- `op.cod` - codomain (i.e., $Y$)

and the operator itself provides two central methods:

- `apply(x)` for the forward action,
- `rapply(y)` for the adjoint action.

So the expected geometry is

$$
\texttt{apply}: X \to Y,
\qquad
\texttt{rapply}: Y \to X.
$$

In finite-dimensional real or complex spaces, `rapply` corresponds to the adjoint operator

$$
A^* : Y \to X.
$$
$$
\langle Ax, y \rangle_Y = \langle x, A^* y \rangle_X.
$$


In [ ]:
import numpy as np

from spacecore.backend import Context, NumpyOps
from spacecore.space import VectorSpace
from spacecore.linop import (
    DenseLinOp,
    SparseLinOp,
    BlockDiagonalLinOp,
    StackedLinOp,
    SumToSingleLinOp,
)

ctx = Context(NumpyOps(), dtype=np.float64, enable_checks=True)

X = VectorSpace((2,), ctx=ctx)
Y = VectorSpace((3,), ctx=ctx)

print(X)
print(Y)


## 3. `DenseLinOp`

`DenseLinOp` is the standard dense linear operator.

It represents a linear map stored by a dense array. If

$$
A : X \to Y,
$$

then the dense storage is a tensor compatible with the domain and codomain shapes. Conceptually, this is the most direct finite-dimensional representation of a linear map.

Use `DenseLinOp` when the operator is naturally stored as a dense matrix or dense tensor.


In [ ]:
A = np.array([
    [1.0, 2.0],
    [0.0, -1.0],
    [3.0, 1.0],
])

op_dense = DenseLinOp(A, X, Y, ctx=ctx)
print(op_dense)
print('domain =', op_dense.dom)
print('codomain =', op_dense.cod)


### 3.1 Forward application

The method `apply(x)` computes

$$
x \mapsto Ax.
$$

So if \(x \in X\), then `apply(x)` returns an element of the codomain \(Y\).


In [ ]:
x = np.array([1.0, -1.0])
y = op_dense.apply(x)

print('x =', x)
print('Ax =', y)


### 3.2 Adjoint application

The method `rapply(y)` computes the adjoint action

$$
y \mapsto A^* y.
$$

So if \(y \in Y\), then `rapply(y)` returns an element of the domain \(X\).


In [ ]:
z = np.array([1.0, 2.0, 3.0])
w = op_dense.rapply(z)

print('z =', z)
print('A* z =', w)


## 4. `SparseLinOp`

`SparseLinOp` represents a linear map stored in sparse form.

This is useful when the operator has many zero entries and dense storage would be wasteful.

Conceptually it still represents the same kind of object,

$$
A : X \to Y,
$$

but the internal storage is sparse rather than dense.

Use `SparseLinOp` when sparsity is part of the operator structure.


The example below uses a SciPy sparse matrix under the NumPy context.
Depending on your local installation, sparse support may also be available for other backends.


In [ ]:
import scipy.sparse as sp

A_sparse = sp.csr_matrix(np.array([
    [1.0, 0.0],
    [0.0, -1.0],
    [3.0, 0.0],
]))

op_sparse = SparseLinOp(A_sparse, X, Y, ctx=ctx)
print(op_sparse)


In [ ]:
x = np.array([2.0, 1.0])
print('Ax =', op_sparse.apply(x))

y = np.array([1.0, 2.0, 3.0])
print('A* y =', op_sparse.rapply(y))


## 5. Product operators

The library also implements several operators built from families of operators.

These operators act naturally on product spaces and encode block structure explicitly.

This is important when variables are naturally split into components,

$$
x = (x_1, \dots, x_k),
$$

and the linear map respects that decomposition.


## 6. `BlockDiagonalLinOp`

A block-diagonal operator is built from operators

$$
A_i : X_i \to Y_i,
$$

and acts on the product space

$$
X_1 \times \cdots \times X_k
\to
Y_1 \times \cdots \times Y_k
$$

by applying each block independently:

$$
(x_1,\dots,x_k) \mapsto (A_1 x_1,\dots,A_k x_k).
$$

This is the natural operator for independent block actions.


In [ ]:
X1 = VectorSpace((2,), ctx=ctx)
Y1 = VectorSpace((2,), ctx=ctx)
X2 = VectorSpace((3,), ctx=ctx)
Y2 = VectorSpace((3,), ctx=ctx)

A1 = DenseLinOp(np.array([[1.0, 0.0], [0.0, 2.0]]), X1, Y1, ctx=ctx)
A2 = DenseLinOp(np.eye(3), X2, Y2, ctx=ctx)

op_block = BlockDiagonalLinOp.from_operators((A1, A2))
print(op_block)


In [ ]:
x_block = (
    np.array([1.0, 2.0]),
    np.array([3.0, 4.0, 5.0]),
)

y_block = op_block.apply(x_block)
print('input  =', x_block)
print('output =', y_block)


## 7. `StackedLinOp`

`StackedLinOp` stacks several operators with the same domain

$$
A_i : X \to Y_i,
$$

into one operator

$$
X \to Y_1 \times \cdots \times Y_k,
$$

defined by

$$
x \mapsto (A_1 x, \dots, A_k x).
$$

So one input is sent to multiple output blocks.


In [ ]:
X = VectorSpace((2,), ctx=ctx)
Y1 = VectorSpace((2,), ctx=ctx)
Y2 = VectorSpace((3,), ctx=ctx)

A1 = DenseLinOp(np.array([[1.0, 0.0], [0.0, 1.0]]), X, Y1, ctx=ctx)
A2 = DenseLinOp(np.array([[1.0, 2.0], [0.0, -1.0], [3.0, 1.0]]), X, Y2, ctx=ctx)

op_stacked = StackedLinOp.from_operators((A1, A2))
print(op_stacked)


In [ ]:
x = np.array([1.0, -1.0])
y = op_stacked.apply(x)

print('x =', x)
print('stacked output =', y)


The adjoint of a stacked operator goes in the opposite direction:

$$
(y_1,\dots,y_k) \mapsto A_1^* y_1 + \cdots + A_k^* y_k.
$$

So the adjoint combines contributions from all output blocks back into the common domain.


In [ ]:
y_in = (
    np.array([1.0, 2.0]),
    np.array([3.0, 4.0, 5.0]),
)

print('adjoint output =', op_stacked.rapply(y_in))


## 8. `SumToSingleLinOp`

`SumToSingleLinOp` combines operators with a common codomain

$$
A_i : X_i \to Y,
$$

into one operator

$$
X_1 \times \cdots \times X_k \to Y,
$$

defined by

$$
(x_1,\dots,x_k) \mapsto A_1 x_1 + \cdots + A_k x_k.
$$

So several input blocks contribute to one common output.


In [ ]:
X1 = VectorSpace((2,), ctx=ctx)
X2 = VectorSpace((3,), ctx=ctx)
Y = VectorSpace((2,), ctx=ctx)

A1 = DenseLinOp(np.array([[1.0, 0.0], [0.0, 1.0]]), X1, Y, ctx=ctx)
A2 = DenseLinOp(np.array([[1.0, 2.0, 0.0], [0.0, -1.0, 1.0]]), X2, Y, ctx=ctx)

op_sum = SumToSingleLinOp.from_operators((A1, A2))
print(op_sum)


In [ ]:
x_in = (
    np.array([1.0, 2.0]),
    np.array([3.0, 4.0, 5.0]),
)

print('sum-to-single output =', op_sum.apply(x_in))


The adjoint sends one output element back to a tuple of domain contributions:

$$
y \mapsto (A_1^* y, \dots, A_k^* y).
$$

So `SumToSingleLinOp` and `StackedLinOp` are natural adjoint-shaped companions.


In [ ]:
y = np.array([1.0, -1.0])
print('adjoint output =', op_sum.rapply(y))


## 9. How to define a linear operator

There are two main patterns.

### Pattern 1: use an existing concrete operator class

This is what most users will do.

Examples:

- use `DenseLinOp(A, dom, cod)` when you have dense storage,
- use `SparseLinOp(A, dom, cod)` when you have sparse storage,
- use the product constructors `from_operators(...)` when you want structured block operators.

### Pattern 2: define a new subclass of `LinOp`

This is useful when the linear map has a special structure and you do not want to materialize dense or sparse storage.

Then you typically define:

- the domain,
- the codomain,
- `apply`,
- `rapply`.

So a custom operator is defined by its action, not necessarily by an explicit matrix.


In [ ]:
from spacecore.linop import LinOp

class ScaleLinOp(LinOp):
    def __init__(self, alpha, dom, cod=None, ctx=None):
        if cod is None:
            cod = dom
        super().__init__(dom=dom, cod=cod, ctx=ctx)
        self.alpha = alpha

    def apply(self, x):
        return self.cod.scale(self.alpha, x)

    def rapply(self, y):
        return self.dom.scale(self.alpha, y)


X = VectorSpace((3,), ctx=ctx)
op_scale = ScaleLinOp(2.0, X, ctx=ctx)
print(op_scale.apply(np.array([1.0, 2.0, 3.0])))


The previous example is intentionally simple. Its point is to illustrate that `LinOp` is fundamentally an abstraction of linear action

$$
x \mapsto Ax,
$$

rather than merely an array container.


## 10. Why `LinOp` matters

Without the `LinOp` abstraction, one would manipulate raw dense or sparse arrays directly.
Then information about domain, codomain, block structure, and adjoint action would remain implicit.

With `LinOp`, the library makes those geometric features explicit.
This improves:

- readability,
- correctness,
- composability,
- backend independence.


## 11. Summary

A `LinOp` is the abstraction of a linear map between two spaces.

It stores:

- a domain,
- a codomain,
- a context,
- forward and adjoint actions.

Current implemented operator types are:

- `DenseLinOp`,
- `SparseLinOp`,
- `BlockDiagonalLinOp`,
- `StackedLinOp`,
- `SumToSingleLinOp`.

So the role of `LinOp` is to turn raw matrix-like storage and structured block maps into explicit geometric linear operators inside the library.
